# 04 — Multi-Chain Routing

Route user queries to specialised expert chains based on detected topic.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Define Router and Expert Prompts

In [ ]:
class RouteDecision(BaseModel):
    """Decides which expert chain should handle the query."""
    topic: str = Field(description="One of: 'coding', 'science', 'history', 'general'")

EXPERT_PROMPTS = {
    "coding": ChatPromptTemplate.from_template(
        "You are an expert software engineer. Give a clear, practical answer "
        "with code examples when appropriate.\n\nQuestion: {question}"
    ),
    "science": ChatPromptTemplate.from_template(
        "You are a science educator. Explain concepts clearly, using analogies "
        "and real-world examples.\n\nQuestion: {question}"
    ),
    "history": ChatPromptTemplate.from_template(
        "You are a historian. Provide accurate, engaging answers with relevant "
        "dates and context.\n\nQuestion: {question}"
    ),
    "general": ChatPromptTemplate.from_template(
        "You are a helpful assistant. Answer the question clearly and concisely."
        "\n\nQuestion: {question}"
    ),
}

## Route and Answer

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
output_parser = StrOutputParser()

router_prompt = ChatPromptTemplate.from_template(
    "Classify this question into exactly one category: coding, science, history, or general.\n\n"
    "Question: {question}"
)
router_chain = router_prompt | llm.with_structured_output(RouteDecision)

questions = [
    "How do I reverse a linked list in Python?",
    "Why is the sky blue?",
    "What caused the fall of the Roman Empire?",
    "What's a good recipe for banana bread?",
]

for q in questions:
    route = router_chain.invoke({"question": q})
    topic = route.topic if route.topic in EXPERT_PROMPTS else "general"
    expert_chain = EXPERT_PROMPTS[topic] | llm | output_parser
    answer = expert_chain.invoke({"question": q})
    print(f"Q: {q}\nRouted to: {topic}\nA: {answer[:200]}...\n")